# Result analysis

This notebook loads one or more evaluation JSON files and builds a paper-style benchmark table. Edit `result_files` to add more models.

## Paper-style Columns

`MoE Method`, `PPL ↓`, `LAMBADA`, `BLiMP`, `CBT`, `HellaSwag`, `PIQA`, `ARC-Easy`, `RACE`, `SIQA`, `CommonSenseQA`, `AVG Acc ↑`, `AVG Rank ↓`

In [2]:
from pathlib import Path
import json
from html import escape
from zipfile import ZIP_DEFLATED, ZipFile

import pandas as pd
from IPython.display import HTML

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_colwidth", 120)

# Add as many result files as you want here.
# Example: {"smoe": "path/to/smoe.json", "xmoe": "path/to/xmoe.json"}
result_files = {
    "Model-1000": "result-model-1000.pth.json",
}

notebook_dir = Path("/mnt/d/workspace/LibMoE/language_modeling/assets")
repo_root = notebook_dir.parents[1]

def resolve_result_path(path):
    path = Path(path).expanduser()
    candidates = [path, Path.cwd() / path, notebook_dir / path, repo_root / path]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(f"Could not find result file: {path}")

loaded_results = {}
resolved_result_paths = {}
for model_name, path in result_files.items():
    result_path = resolve_result_path(path)
    with result_path.open("r", encoding="utf-8") as f:
        loaded_results[model_name] = json.load(f)
    resolved_result_paths[model_name] = result_path
    print(f"Loaded {model_name}: {len(loaded_results[model_name])} metrics from {result_path}")

default_model_name = next(iter(loaded_results))
results = loaded_results[default_model_name]
result_path = resolved_result_paths[default_model_name]

Loaded Model-1000: 110 metrics from /mnt/d/workspace/LibMoE/language_modeling/assets/result-model-1000.pth.json


## All Metrics

In [3]:
metrics = pd.DataFrame(
    [
        {
            "model": model_name,
            "metric": key,
            "group": key.split("/")[0],
            "name": "/".join(key.split("/")[1:]) if "/" in key else key,
            "value": value,
        }
        for model_name, model_results in loaded_results.items()
        for key, value in model_results.items()
    ]
).sort_values(["model", "group", "metric"]).reset_index(drop=True)

display(metrics)
metrics.to_csv("result-all-metrics.csv", index=False)

,model,metric,group,name,value
0,Model-1000,ai2arc/accuracy/ARC-Challenge,ai2arc,accuracy/ARC-Challenge,0.197425
1,Model-1000,ai2arc/accuracy/ARC-Easy,ai2arc,accuracy/ARC-Easy,0.260888
2,Model-1000,ai2arc/accuracy/group_average,ai2arc,accuracy/group_average,0.229156
3,Model-1000,ai2arc/accuracy/seq_average,ai2arc,accuracy/seq_average,0.239943
4,Model-1000,blimp/accuracy/adjunct_island,blimp,accuracy/adjunct_island,0.718000
5,Model-1000,blimp/accuracy/anaphor_gender_agreement,blimp,accuracy/anaphor_gender_agreement,0.441000
6,Model-1000,blimp/accuracy/anaphor_number_agreement,blimp,accuracy/anaphor_number_agreement,0.692000
7,Model-1000,blimp/accuracy/animate_subject_passive,blimp,accuracy/animate_subject_passive,0.663000
8,Model-1000,blimp/accuracy/animate_subject_trans,blimp,accuracy/animate_subject_trans,0.692000
9,Model-1000,blimp/accuracy/causative,blimp,accuracy/causative,0.494000


## Benchmark Metrics

In [4]:
benchmark_accuracy_metrics = {
    "LAMBADA": "lambada/accuracy/total",
    "BLiMP": "blimp/accuracy/group_average",
    "CBT": "cbt/accuracy/group_average",
    "HellaSwag": "hellaswag/accuracy/group_average",
    "PIQA": "piqa/accuracy/group_average",
    "ARC-Easy": "ai2arc/accuracy/group_average",
    "RACE": "race/accuracy/group_average",
    "SIQA": "siqa/accuracy/group_average",
    "CommonSenseQA": "commonsenseqa/accuracy/group_average",
}

benchmark_accuracy = pd.DataFrame(
    [
        {
            "model": model_name,
            "benchmark": label,
            "metric": metric,
            "accuracy": model_results[metric],
        }
        for model_name, model_results in loaded_results.items()
        for label, metric in benchmark_accuracy_metrics.items()
        if metric in model_results
    ]
).sort_values(["benchmark", "accuracy"], ascending=[True, False])

display(benchmark_accuracy)
benchmark_accuracy.to_csv("result-benchmark-accuracy.csv", index=False)

,model,benchmark,metric,accuracy
5,Model-1000,ARC-Easy,ai2arc/accuracy/group_average,0.229156
1,Model-1000,BLiMP,blimp/accuracy/group_average,0.589866
2,Model-1000,CBT,cbt/accuracy/group_average,0.444664
8,Model-1000,CommonSenseQA,commonsenseqa/accuracy/group_average,0.208845
3,Model-1000,HellaSwag,hellaswag/accuracy/group_average,0.245071
0,Model-1000,LAMBADA,lambada/accuracy/total,0.002135
4,Model-1000,PIQA,piqa/accuracy/group_average,0.521763
6,Model-1000,RACE,race/accuracy/group_average,0.239219
7,Model-1000,SIQA,siqa/accuracy/group_average,0.344933


## Paper-style Table

In [5]:
def pct(model_results, metric_name):
    return model_results[metric_name] * 100

def benchmark_accuracy_pct(model_results, prefix, fallback_metric=None):
    group_average_metric = f"{prefix}/accuracy/group_average"
    if group_average_metric in model_results:
        return pct(model_results, group_average_metric)
    if fallback_metric and fallback_metric in model_results:
        return pct(model_results, fallback_metric)
    raise KeyError(f"No group_average metric found for {prefix}")

def excel_column_name(number):
    name = ""
    while number:
        number, remainder = divmod(number - 1, 26)
        name = chr(65 + remainder) + name
    return name

def write_simple_xlsx(df, path, sheet_name="Results"):
    def cell_xml(row_idx, col_idx, value):
        cell_ref = f"{excel_column_name(col_idx)}{row_idx}"
        if pd.isna(value):
            return f'<c r="{cell_ref}"/>'
        if isinstance(value, (int, float)) and not isinstance(value, bool):
            return f'<c r="{cell_ref}"><v>{value}</v></c>'
        return f'<c r="{cell_ref}" t="inlineStr"><is><t>{escape(str(value))}</t></is></c>'

    rows_xml = []
    all_rows = [list(df.columns)] + df.values.tolist()
    for row_idx, row in enumerate(all_rows, start=1):
        cells = "".join(cell_xml(row_idx, col_idx, value) for col_idx, value in enumerate(row, start=1))
        rows_xml.append(f'<row r="{row_idx}">{cells}</row>')

    sheet_xml = "".join([
        '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>',
        '<worksheet xmlns="http://schemas.openxmlformats.org/spreadsheetml/2006/main">',
        '<sheetData>',
        "".join(rows_xml),
        '</sheetData></worksheet>',
    ])
    workbook_xml = "".join([
        '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>',
        '<workbook xmlns="http://schemas.openxmlformats.org/spreadsheetml/2006/main" ',
        'xmlns:r="http://schemas.openxmlformats.org/officeDocument/2006/relationships">',
        f'<sheets><sheet name="{escape(sheet_name, quote=True)}" sheetId="1" r:id="rId1"/></sheets>',
        '</workbook>',
    ])

    with ZipFile(path, "w", ZIP_DEFLATED) as archive:
        archive.writestr("[Content_Types].xml", '<?xml version="1.0" encoding="UTF-8" standalone="yes"?><Types xmlns="http://schemas.openxmlformats.org/package/2006/content-types"><Default Extension="rels" ContentType="application/vnd.openxmlformats-package.relationships+xml"/><Default Extension="xml" ContentType="application/xml"/><Override PartName="/xl/workbook.xml" ContentType="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet.main+xml"/><Override PartName="/xl/worksheets/sheet1.xml" ContentType="application/vnd.openxmlformats-officedocument.spreadsheetml.worksheet+xml"/></Types>')
        archive.writestr("_rels/.rels", '<?xml version="1.0" encoding="UTF-8" standalone="yes"?><Relationships xmlns="http://schemas.openxmlformats.org/package/2006/relationships"><Relationship Id="rId1" Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/officeDocument" Target="xl/workbook.xml"/></Relationships>')
        archive.writestr("xl/workbook.xml", workbook_xml)
        archive.writestr("xl/_rels/workbook.xml.rels", '<?xml version="1.0" encoding="UTF-8" standalone="yes"?><Relationships xmlns="http://schemas.openxmlformats.org/package/2006/relationships"><Relationship Id="rId1" Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/worksheet" Target="worksheets/sheet1.xml"/></Relationships>')
        archive.writestr("xl/worksheets/sheet1.xml", sheet_xml)

paper_rows = []
for model_name, model_results in loaded_results.items():
    paper_rows.append(
        {
            "MoE Method": model_name,
            "PPL ↓": model_results["val/perplexity"],
            "LAMBADA": pct(model_results, "lambada/accuracy/total"),
            "BLiMP": benchmark_accuracy_pct(model_results, "blimp"),
            "CBT": benchmark_accuracy_pct(model_results, "cbt"),
            "HellaSwag": benchmark_accuracy_pct(model_results, "hellaswag"),
            "PIQA": benchmark_accuracy_pct(model_results, "piqa"),
            "ARC-Easy": benchmark_accuracy_pct(model_results, "ai2arc", "ai2arc/accuracy/ARC-Easy"),
            "RACE": benchmark_accuracy_pct(model_results, "race"),
            "SIQA": benchmark_accuracy_pct(model_results, "siqa"),
            "CommonSenseQA": benchmark_accuracy_pct(model_results, "commonsenseqa"),
        }
    )

accuracy_columns = ["LAMBADA", "BLiMP", "CBT", "HellaSwag", "PIQA", "ARC-Easy", "RACE", "SIQA", "CommonSenseQA"]
paper_table = pd.DataFrame(paper_rows)
paper_table["AVG Acc ↑"] = paper_table[accuracy_columns].mean(axis=1)

rank_columns = ["PPL ↓"] + accuracy_columns
rank_table = pd.DataFrame(index=paper_table.index)
rank_table["PPL ↓"] = paper_table["PPL ↓"].rank(method="average", ascending=True)
for column in accuracy_columns:
    rank_table[column] = paper_table[column].rank(method="average", ascending=False)
paper_table["AVG Rank ↓"] = rank_table[rank_columns].mean(axis=1)
paper_table = paper_table.sort_values("AVG Rank ↓").reset_index(drop=True)

paper_table.to_csv("result-paper-table.csv", index=False)
write_simple_xlsx(paper_table, "result-paper-table.xlsx")

header_labels = ["MoE Method", "PPL ↓", "LAM<br>BADA", "BLiMP", "CBT", "Hella<br>Swag", "PIQA", "ARC-<br>Easy", "RACE", "SIQA", "Common<br>SenseQA", "AVG<br>Acc ↑", "AVG<br>Rank ↓"]
columns = list(paper_table.columns)
def format_table_value(value):
    return f"{value:.2f}" if isinstance(value, (int, float)) else str(value)
header_html = "".join(f"<th>{label}</th>" for label in header_labels)
body_html = "".join(
    "<tr>" + "".join(f"<td>{format_table_value(row[column])}</td>" for column in columns) + "</tr>"
    for _, row in paper_table.iterrows()
)
display(HTML(f"""
<style>
.paper-result-table {{
    border-collapse: collapse;
    border-top: 2px solid #111;
    border-bottom: 2px solid #111;
    font-family: 'Times New Roman', Times, serif;
    font-size: 18px;
    margin: 12px 0;
    min-width: 100%;
}}
.paper-result-table th {{
    border-bottom: 1px solid #333;
    font-size: 20px;
    font-weight: 700;
    line-height: 1.05;
    padding: 6px 12px 8px;
    text-align: center;
    white-space: nowrap;
}}
.paper-result-table td {{
    padding: 5px 12px;
    text-align: center;
}}
.paper-result-table td:first-child {{
    text-align: left;
}}
</style>
<table class="paper-result-table">
    <thead><tr>{header_html}</tr></thead>
    <tbody>{body_html}</tbody>
</table>
"""))

MoE Method,PPL ↓,LAMBADA,BLiMP,CBT,HellaSwag,PIQA,ARC-Easy,RACE,SIQA,CommonSenseQA,AVGAcc ↑,AVGRank ↓
Model-1000,89.70,0.21,58.99,44.47,24.51,52.18,22.92,23.92,34.49,20.88,31.40,1.00
